# Causal A/B Test Analysis

Executed walkthrough covering hypothesis testing, Bayesian posterior inference, CUPED, propensity matching, and subgroup analysis with Bonferroni correction.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
from src.analysis import *
from src.bayesian import bayesian_ab_summary
raw = load_data('../data/raw/ab_data.csv')
audit = audit_data(raw)
df = clean_data(raw)
audit, df.shape

({'n_total': 298306,
  'n_duplicates': 3828,
  'n_mismatches': 3878,
  'mismatch_rate': np.float64(0.013),
  'conversion_overall': np.float64(0.1191),
  'group_balance': {'control': 149566, 'treatment': 148740}},
 (290650, 5))

## Frequentist hypothesis test

In [2]:
naive = naive_comparison(df)
ztest = ztest_conversion(df)
power = power_analysis(df)
naive, ztest, power

({'control_rate': np.float64(0.1195),
  'treatment_rate': np.float64(0.1187),
  'absolute_lift': np.float64(-0.0007),
  'relative_lift': np.float64(-0.0059),
  'n_control': 145680,
  'n_treatment': 144970},
 {'z_stat': -0.5866,
  'p_value': 0.5575,
  'significant': False,
  'ci_95': (np.float64(-0.0031), np.float64(0.0017))},
 {'effect_size': np.float64(-0.0025),
  'required_n_per_group': 2573321,
  'actual_n_per_group': 145680,
  'actual_power': 0.1023,
  'adequately_powered': False})

## Bayesian posterior

In [3]:
bayes = bayesian_ab_summary(df)
bayes

{'prior': (1.0, 1.0),
 'posterior_control': (17403.0, 128279.0),
 'posterior_treatment': (17216.0, 127756.0),
 'mean_control': 0.119459,
 'mean_treatment': 0.118754,
 'ci_95_control': (0.11779834270100303, 0.12112919618127417),
 'ci_95_treatment': (0.1170937145991827, 0.12042418120009094),
 'prob_treatment_better': 0.2797,
 'prob_control_better': 0.7204,
 'loss_if_ship_treatment': 0.00091,
 'loss_if_keep_control': 0.000209}

## CUPED variance reduction

In [4]:
cuped = cuped_adjustment(df)
cuped

{'theta': np.float64(0.00017699),
 'x_bar': 13.9029,
 'var_orig': 0.10518516,
 'var_adj': 0.10518434,
 'var_reduction_pct': 0.001,
 'se_orig': 0.00120162,
 'se_adj': 0.00120163,
 'se_reduction_pct': -0.001,
 'mean_ctrl_orig': 0.119454,
 'mean_treat_orig': 0.118749,
 'mean_ctrl_adj': 0.119452,
 'mean_treat_adj': 0.118751,
 'lift_orig': -0.000705,
 'lift_adj': -0.000701,
 't_stat': -0.5834,
 'p_value': 0.5596,
 'significant': False,
 'n_control': 145680,
 'n_treatment': 144970}

## Propensity score matching

In [5]:
matched = propensity_score_match(df)
matched_summary = naive_comparison(matched)
matched_test = ztest_conversion(matched)
matched.shape, matched_summary, matched_test

((336, 8),
 {'control_rate': np.float64(0.1429),
  'treatment_rate': np.float64(0.1131),
  'absolute_lift': np.float64(-0.0298),
  'relative_lift': np.float64(-0.2083),
  'n_control': 168,
  'n_treatment': 168},
 {'z_stat': -0.8165,
  'p_value': 0.4142,
  'significant': False,
  'ci_95': (np.float64(-0.1011), np.float64(0.0416))})

## Subgroup analysis with Bonferroni correction

In [6]:
subgroups = subgroup_analysis(df)
subgroups[['subgroup_type', 'subgroup', 'lift', 'p_raw', 'p_bonferroni', 'significant']]

,subgroup_type,subgroup,lift,p_raw,p_bonferroni,significant
0,hour_bucket,Afternoon (12–17),-0.0020,0.3172,1.0000,False
1,hour_bucket,Evening (18–23),-0.0032,0.1548,0.9288,False
2,hour_bucket,Morning (6–11),0.0034,0.1167,0.7002,False
3,hour_bucket,Night (0–5),-0.0026,0.6551,1.0000,False
4,day_type,Weekday,-0.0004,0.7640,1.0000,False
5,day_type,Weekend,-0.0015,0.5265,1.0000,False


## Decision

Across frequentist, Bayesian, matched, CUPED, and subgroup checks, the redesign does not produce evidence strong enough to ship.